In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


# ── 公共工具函数 ──────────────────────────────────────────────

def compute_max_accuracy(csv_path: Path) -> dict | None:
    """读取 CSV，返回最大 accuracy，若无 accuracy 列则返回 None。"""
    try:
        df = pd.read_csv(csv_path)
        if "accuracy" not in df.columns:
            return None
        max_acc = df["accuracy"].max()
        return {"max_accuracy": max_acc}
    except Exception as e:
        print(f"[错误] {csv_path} -> {e}")
        return None


def compute_aulc(csv_path: Path) -> dict | None:
    """
    读取 CSV，计算 AULC（曲线下面积）和归一化 AULC。
    若有 round 列，以 round 为 x 轴；否则以行索引为 x 轴。
    返回包含 max_accuracy / aulc / normalized_aulc 的字典，若失败返回 None。
    """
    try:
        df = pd.read_csv(csv_path)
        if "accuracy" not in df.columns:
            return None

        if "round" in df.columns:
            curve_df = df[["round", "accuracy"]].dropna().sort_values("round")
            x = curve_df["round"].to_numpy(dtype=float)
        else:
            curve_df = df[["accuracy"]].dropna()
            x = np.arange(len(curve_df), dtype=float)

        y = curve_df["accuracy"].to_numpy(dtype=float)

        if len(y) == 0:
            return None

        if len(y) == 1:
            aulc = 0.0
            normalized_aulc = float(y[0])
        else:
            aulc = float(np.trapezoid(y, x)) # aulc
            span = x[-1] - x[0]
            normalized_aulc = aulc / span if span > 0 else float(y.mean())

        return {
            "max_accuracy": float(y.max()),
            "aulc": aulc,
            "normalized_aulc": normalized_aulc,
        }
    except Exception as e:
        print(f"[错误] {csv_path} -> {e}")
        return None


def scan_dir_max_accuracy(root: Path, pattern: str = "**/*.csv",
                           extra_key: str | None = None) -> pd.DataFrame:
    """
    遍历 root 下匹配 pattern 的 CSV，收集最大 accuracy。
    extra_key: 若指定，则把相对父目录名也写入结果（如 client_folder）。
    """
    rows = []
    for csv_file in sorted(root.glob(pattern)):
        result = compute_max_accuracy(csv_file)
        if result is None:
            continue
        row = {"csv_file": csv_file.name, **result}
        if extra_key:
            row[extra_key] = csv_file.parent.name
        rows.append(row)
    return pd.DataFrame(rows)


def scan_dir_aulc(root: Path, pattern: str = "**/*.csv",
                  extra_key: str | None = None) -> pd.DataFrame:
    """
    遍历 root 下匹配 pattern 的 CSV，收集 AULC 指标。
    extra_key: 若指定，则把相对父目录名也写入结果（如 client_folder）。
    """
    rows = []
    for csv_file in sorted(root.glob(pattern)):
        result = compute_aulc(csv_file)
        if result is None:
            continue
        row = {"csv_file": csv_file.name, **result}
        if extra_key:
            row[extra_key] = csv_file.parent.name
        rows.append(row)
    return pd.DataFrame(rows)


print("✅ 公共函数已加载：compute_max_accuracy / compute_aulc / scan_dir_max_accuracy / scan_dir_aulc")


✅ 公共函数已加载：compute_max_accuracy / compute_aulc / scan_dir_max_accuracy / scan_dir_aulc


In [6]:
# ── rank_analyze_experiment：最大 accuracy + AULC ─────────────

rank_dir = Path("/Users/chensj/Documents/GitHub/paper_experiment_record/onelora/rank_analyze_experiment")

# 最大 accuracy（按 client_folder 分组）
max_df = scan_dir_max_accuracy(rank_dir, pattern="client_*_mnist_rank_analyze_topsis/*.csv",
                                extra_key="client_folder")
max_df = max_df.sort_values("max_accuracy", ascending=False).reset_index(drop=True)
max_df.to_csv(rank_dir / "accuracy_max_summary.csv", index=False)
print("=== Max Accuracy ===")
print(max_df.to_string(index=False))

print()

# AULC（按 client_folder 分组）
aulc_df = scan_dir_aulc(rank_dir, pattern="client_*_mnist_rank_analyze_topsis/*.csv",
                         extra_key="client_folder")
aulc_df = aulc_df.sort_values("normalized_aulc", ascending=False).reset_index(drop=True)
aulc_df.to_csv(rank_dir / "accuracy_aulc_summary.csv", index=False)
print("=== AULC ===")
print(aulc_df.to_string(index=False))


=== Max Accuracy ===
csv_file  max_accuracy                      client_folder
0.50.csv        0.9386 client_8_mnist_rank_analyze_topsis
0.50.csv        0.9368 client_7_mnist_rank_analyze_topsis
0.90.csv        0.9351 client_8_mnist_rank_analyze_topsis
0.10.csv        0.9349 client_7_mnist_rank_analyze_topsis
0.90.csv        0.9348 client_7_mnist_rank_analyze_topsis
0.40.csv        0.9347 client_8_mnist_rank_analyze_topsis
0.20.csv        0.9340 client_8_mnist_rank_analyze_topsis
0.10.csv        0.9340 client_3_mnist_rank_analyze_topsis
0.10.csv        0.9339 client_6_mnist_rank_analyze_topsis
1.00.csv        0.9333 client_3_mnist_rank_analyze_topsis
0.10.csv        0.9332 client_5_mnist_rank_analyze_topsis
0.10.csv        0.9330 client_1_mnist_rank_analyze_topsis
0.10.csv        0.9328 client_2_mnist_rank_analyze_topsis
0.10.csv        0.9327 client_0_mnist_rank_analyze_topsis
0.10.csv        0.9325 client_8_mnist_rank_analyze_topsis
0.90.csv        0.9325 client_6_mnist_rank_analyze_

In [9]:
# ── main_experiment：AULC ────────────────────────────────────

main_dir = Path("/Users/chensj/Documents/GitHub/paper_experiment_record/onelora/main_experiment")

result_df = scan_dir_aulc(main_dir, pattern="*.csv")

# 格式化为百分比（保留 1 位小数）
result_df["max_accuracy"]     = (result_df["max_accuracy"]     * 100).round(1)
result_df["normalized_aulc"]  = (result_df["normalized_aulc"]  * 100).round(1)
result_df["aulc"]             =  result_df["aulc"].round(4)

result_df = result_df.sort_values("csv_file").reset_index(drop=True)

output_file = main_dir / "main_experiment_aulc_summary.csv"
result_df.to_csv(output_file, index=False)

print(f"结果已保存到: {output_file}\n")
print(result_df.to_string(index=False))


结果已保存到: /Users/chensj/Documents/GitHub/paper_experiment_record/onelora/main_experiment/main_experiment_aulc_summary.csv

                                           csv_file  max_accuracy     aulc  normalized_aulc
          flexlora_cifar10_homo_all_participate.csv          31.0  49.3750             24.8
          flexlora_cinic10_homo_all_participate.csv          20.3  23.5946             17.2
           flexlora_fmnist_homo_all_participate.csv          86.6  82.5956             83.4
flexlora_fmnist_homo_same_param_all_participate.csv          77.2  65.1884             65.8
         flexlora_fmnist_manual_all_participate.csv          85.0  80.5704             81.4
            flexlora_mnist_homo_all_participate.csv          96.4  92.6126             93.5
 flexlora_mnist_homo_same_param_all_participate.csv          95.7  91.9373             92.9
          flexlora_mnist_manual_all_participate.csv          93.7  89.3802             90.3
                         rbla_cifar10_homo_rank.csv